<a href="https://colab.research.google.com/github/KenettyAnderson/mestre-rpg-voz-gpt/blob/main/Mestre_de_RPG_por_Voz_com_GPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
# 1. INSTALAÇÃO DAS BIBLIOTECAS (Rode esta célula primeiro no Colab)
!pip install git+https://github.com/openai/whisper.git -q
!pip install openai gTTS -q

# 2. IMPORTS E CONFIGURAÇÕES INICIAIS
import os
import whisper
from gtts import gTTS
from IPython.display import Audio, display, Javascript, clear_output
from google.colab import output
from base64 import b64decode
from openai import OpenAI # Importa a nova classe OpenAI

# Coloque sua API Key da OpenAI aqui
os.environ['OPENAI_API_KEY'] = 'SUA_API_KEY_AQUI'

# Inicializa o cliente da OpenAI com a nova API
client = OpenAI(api_key=os.environ.get('OPENAI_API_KEY'))

# Carrega o modelo do Whisper (usando o 'base' para ser mais rápido, mas pode usar o 'small')
print("Carregando o modelo Whisper... (Isso pode levar alguns segundos)")
modelo_whisper = whisper.load_model("small")
language = 'pt'

# 3. FUNÇÃO DE GRAVAÇÃO DE ÁUDIO (O JavaScript mágico do Colab)
RECORD = """
const sleep  = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})
var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async () =>{
    blob = new Blob(chunks)
    text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""

def gravar_audio(segundos=7):
  print(f"\n🎙️ Ouvindo por {segundos} segundos... Fale agora!")
  display(Javascript(RECORD))
  js_result = output.eval_js('record(%s)' % (segundos * 1000))
  audio = b64decode(js_result.split(',')[1])
  file_name = 'audio_jogador.wav'
  with open(file_name, 'wb') as f:
    f.write(audio)
  return f'/content/{file_name}'

# 4. CONFIGURAÇÃO DA AVENTURA (O Segredo do Mestre)
system_prompt = """
Você é um Mestre de RPG de mesa num cenário de fantasia medieval.
Sua missão é narrar a aventura e reagir às ações do jogador de forma dramática.
Regras:
1. Descreva o ambiente e termine sempre perguntando: "O que você faz?".
2. Mantenha as respostas muito curtas e diretas (máximo 2 parágrafos).
3. Seja criativo nas consequências das ações do jogador.
"""

# Iniciamos a memória do jogo com as regras do Mestre
historico_da_aventura = [
    {"role": "system", "content": system_prompt}
]

# 5. O LOOP DO JOGO (A mágica acontece aqui)
print("\n⚔️ BEM-VINDO À AVENTURA! ⚔️")
print("Diga 'Sair do jogo' quando quiser encerrar.\n")

# Para dar o pontapé inicial, o Mestre fala primeiro:
historico_da_aventura.append({"role": "user", "content": "Comece a aventura narrando onde eu estou."})

while True:
    # --- PENSAR (ChatGPT) ---
    print("🧠 O Mestre está pensando...")
    # Usa o novo cliente e o método `chat.completions.create`
    resposta_api = client.chat.completions.create(
        model="gpt-3.5-turbo", # Pode mudar para gpt-4 se quiser
        messages=historico_da_aventura
    )

    fala_do_mestre = resposta_api.choices[0].message.content

    # Salva a fala do Mestre na memória
    historico_da_aventura.append({"role": "assistant", "content": fala_do_mestre})

    # --- FALAR (gTTS) ---
    print(f"🧙‍♂️ Mestre: {fala_do_mestre}")
    arquivo_voz_mestre = "/content/voz_mestre.wav"
    gtts_object = gTTS(text=fala_do_mestre, lang=language, slow=False)
    gtts_object.save(arquivo_voz_mestre)
    display(Audio(arquivo_voz_mestre, autoplay=True))

    # --- OUVIR O JOGADOR ---
    arquivo_gravado = gravar_audio(segundos=7) # Tempo para o jogador falar

    # --- TRANSCREVER (Whisper) ---
    print("📝 Transcrevendo sua ação...")
    resultado_transcricao = modelo_whisper.transcribe(arquivo_gravado, fp16=False, language=language)
    acao_do_jogador = resultado_transcricao["text"].strip()

    print(f"🗡️ Jogador: {acao_do_jogador}")

    # Condição de parada
    if "sair" in acao_do_jogador.lower() or "encerrar" in acao_do_jogador.lower():
        print("\nFim da aventura. Até a próxima!")
        break

    # Salva a ação do jogador na memória
    historico_da_aventura.append({"role": "user", "content": acao_do_jogador})

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Carregando o modelo Whisper... (Isso pode levar alguns segundos)

⚔️ BEM-VINDO À AVENTURA! ⚔️
Diga 'Sair do jogo' quando quiser encerrar.

🧠 O Mestre está pensando...


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}